# Colab GPU persistence test in VS Code

[SmolLM2-360M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) is a 360M-parameter chat model. Its checkpoint takes about 727 MB in Drive. This notebook requires CUDA and runs inference in float16.

1. In VS Code, use **Select Kernel → Colab → New Colab Server → GPU → T4**. Choose **Latest**, then **Python 3 (ipykernel)**.
2. Run **Setup** and **Save and generate**. Approve Google's Drive prompt and wait for **Drive flush finished**.
3. Use **Colab → Remove Server** for the first test server. Create a new T4 server, then run **Setup** and **Reload and verify**. The test uses a fresh VM because restarting the Jupyter kernel can retain runtime files.

If cells stop starting after replacing the server, reopen this notebook in a new VS Code window, then select the replacement server under **Select Kernel → Colab**.

### Setup · Run on both GPU runtimes

The latest managed Colab runtime supplies PyTorch and Transformers.

In [ ]:
import json
import os
import socket

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Attach a Colab GPU kernel before running this notebook.")

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name} | VM: {socket.gethostname()}", flush=True)
drive.mount("/content/drive")

from pathlib import Path

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
revision = "a10cc1512eabd3dde888204e902eca88bddb4951"
drive_root = Path("/content/drive/MyDrive")
model_dir = drive_root / "colab-models" / model_id.split("/")[-1]
runtime_marker = Path("/content/smollm2-360m-gpu-baseline.marker")

# File sizes in bytes at the pinned Hugging Face revision.
model_files = {
    "config.json": 846,
    "generation_config.json": 132,
    "merges.txt": 466_391,
    "model.safetensors": 723_674_912,
    "special_tokens_map.json": 655,
    "tokenizer.json": 2_104_556,
    "tokenizer_config.json": 3_764,
    "vocab.json": 800_662,
}

def missing_or_incomplete_files(folder):
    bad_files = []
    for name, expected_size in model_files.items():
        path = folder / name
        if not path.is_file() or path.stat().st_size != expected_size:
            bad_files.append(name)
    return bad_files

if not drive_root.is_dir():
    raise RuntimeError("Google Drive did not mount.")

print(f"Model folder: {model_dir}")

boot_id = Path("/proc/sys/kernel/random/boot_id").read_text().strip()
vm_id = f"{socket.gethostname()}:{boot_id}"
receipt_path = model_dir / "gpu-baseline.json"


def generate_from_drive():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    incomplete = missing_or_incomplete_files(model_dir)
    if incomplete:
        raise FileNotFoundError(f"Drive files missing or incomplete: {', '.join(incomplete)}.")

    device = "cuda"
    dtype = torch.float16
    print(f"Loading from {model_dir} on {device}...")

    tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_dir, dtype=dtype, local_files_only=True, use_safetensors=True
    ).to(device).eval()

    prompt = "In one short sentence, why does the Moon have phases?"
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=48,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    reply = tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    if not reply.strip():
        raise RuntimeError("Generation returned an empty reply.")

    print(f"Generated on {model.device} ({gpu_name}): {reply}", flush=True)
    return reply.strip()

### Save and generate · First GPU runtime

Download the pinned Hugging Face revision, copy it to Drive, and generate on the GPU using that copy. The cell then records the VM identity and flushes Drive.

In [ ]:
from shutil import copyfile
from tempfile import TemporaryDirectory

from huggingface_hub import snapshot_download

files_to_copy = missing_or_incomplete_files(model_dir)
if files_to_copy:
    model_dir.mkdir(parents=True, exist_ok=True)
    download_dir = Path("/content") / f"smollm2-360m-download-{revision[:8]}"
    print(f"Downloading {len(files_to_copy)} model file(s) from Hugging Face.", flush=True)
    snapshot_download(
        repo_id=model_id,
        revision=revision,
        local_dir=download_dir,
        allow_patterns=files_to_copy,
        token=False,
    )

    # Publish each file only after its Drive copy is complete.
    with TemporaryDirectory(prefix=".copy-", dir=model_dir) as staging:
        for name in files_to_copy:
            staged_file = Path(staging) / name
            copyfile(download_dir / name, staged_file)
            if staged_file.stat().st_size != model_files[name]:
                raise IOError(f"Incomplete copy: {name}")
            staged_file.replace(model_dir / name)

    print(f"Saved {len(files_to_copy)} file(s) to Drive.")
else:
    print("The complete model is already in Drive.")

reply = generate_from_drive()
runtime_marker.write_text(vm_id)
receipt_path.write_text(json.dumps({
    "revision": revision, "vm_id": vm_id, "gpu": gpu_name, "reply": reply
}, indent=2) + "\n")
drive.flush_and_unmount()
print("GPU baseline passed. Drive flush finished.", flush=True)

### Reload and verify · Fresh GPU runtime

Run Setup, then this cell. It confirms a fresh VM and generates on CUDA from the Drive checkpoint with Hugging Face offline.

In [ ]:
if runtime_marker.exists():
    raise RuntimeError("The baseline marker is still present. Provision a fresh Colab server.")

baseline = json.loads(receipt_path.read_text())
if baseline["revision"] != revision or baseline["vm_id"] == vm_id:
    raise RuntimeError("Expected the pinned model and a different VM from the baseline.")

os.environ["HF_HUB_OFFLINE"] = "1"
reply = generate_from_drive()
print(f"Baseline VM: {baseline['vm_id']} | Reload VM: {vm_id}", flush=True)
print(f"Reply matched baseline: {reply == baseline['reply']}", flush=True)
print("GPU persistence check passed. A fresh VM generated from Google Drive.", flush=True)